# Scenario 3

The same job, run twice. Between the two runs one configuration value was changed.

Compare the two runs in the Spark UI (http://localhost:4040: Stages, Executors, SQL / DataFrame) and explain the difference in chat as:

`Symptom → Where I looked → Evidence (numbers) → Hypothesis → What I would change → How I would verify`

In [1]:
import time
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("m6-scenario-3").getOrCreate()
orders = spark.read.parquet("/opt/data/m3/orders_big")

w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
ranked = orders.withColumn("rn", F.row_number().over(w))

def run(label, shuffle_partitions):
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_partitions))
    t0 = time.perf_counter()
    ranked.write.format("noop").mode("overwrite").save()
    print(f"{label}: spark.sql.shuffle.partitions={shuffle_partitions}: {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 11:04:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

In [2]:
run("warm-up", 200)
run("before", 200)

warm-up: spark.sql.shuffle.partitions=200: 8.8s


[Stage 6:======================================================>  (21 + 1) / 22]

before: spark.sql.shuffle.partitions=200: 5.7s


In [3]:
run("after", 2)

[Stage 9:=============================>                             (1 + 1) / 2]

after: spark.sql.shuffle.partitions=2: 20.0s


## Stop

In [ ]:
spark.stop()